In [1]:
import pandas as pd

# Load the first and second datasets
df1 = pd.read_csv('labeled_data.csv')
df2 = pd.read_csv('Dynamically Generated Hate Dataset v0.2.3.csv')

# Select and rename the necessary columns from the first dataset
df1_selected = df1[['tweet', 'hate_speech']].rename(columns={'tweet': 'text', 'hate_speech': 'label'})

# Concatenate the datasets
merged_df = pd.concat([df2, df1_selected], ignore_index=True)

# Display the merged dataset
merged_df.head()

,Unnamed: 0,acl.id,X1,text,label,type,target,level,split,round.base,annotator,round,acl.id.matched
0,1.0,acl1,1.0,dalits are lowlives,hate,notgiven,notgiven,original,train,1.0,GNZuCtwed3,1,NaN
1,2.0,acl10,2.0,gay people are a burden to our society,hate,notgiven,notgiven,original,train,1.0,GNZuCtwed3,1,NaN
2,3.0,acl100,3.0,Arabs are not welcome,hate,notgiven,notgiven,original,train,1.0,vDe7GN0NrL,1,NaN
3,4.0,acl1000,4.0,I'm not saying we should actually eliminate he...,hate,notgiven,notgiven,original,train,1.0,oemYWm1Tjg,1,NaN
4,5.0,acl10000,5.0,bananas are for black people,hate,notgiven,notgiven,original,test,1.0,QiOKkCi7F8,1,NaN


In [2]:
# List of columns to drop
columns_to_drop = ['Unnamed: 0', 'acl.id', 'X1', 'acl.id.matched', 'round.base', 'round', 'annotator']

# Drop the columns
merged_df = merged_df.drop(columns=columns_to_drop)

# Display the remaining columns
print("Remaining columns:", merged_df.columns)

Remaining columns: Index(['text', 'label', 'type', 'target', 'level', 'split'], dtype='object')


In [3]:
merged_df.head()

,text,label,type,target,level,split
0,dalits are lowlives,hate,notgiven,notgiven,original,train
1,gay people are a burden to our society,hate,notgiven,notgiven,original,train
2,Arabs are not welcome,hate,notgiven,notgiven,original,train
3,I'm not saying we should actually eliminate he...,hate,notgiven,notgiven,original,train
4,bananas are for black people,hate,notgiven,notgiven,original,test


In [4]:
# Fill missing values for `type`, `target`, `level`, and `split` columns with defaults
merged_df['type'].fillna('notgiven', inplace=True)
merged_df['target'].fillna('notgiven', inplace=True)
merged_df['level'].fillna('original', inplace=True)
merged_df['split'].fillna('train', inplace=True)

# Standardize the `label` column to textual values ('hate', 'nothate')
merged_df['label'] = merged_df['label'].replace({0: 'nothate', 1: 'hate'})

# Display the cleaned dataset
print(merged_df.head())

                                                text label      type  \
0                                dalits are lowlives  hate  notgiven   
1             gay people are a burden to our society  hate  notgiven   
2                              Arabs are not welcome  hate  notgiven   
3  I'm not saying we should actually eliminate he...  hate  notgiven   
4                       bananas are for black people  hate  notgiven   

     target     level  split  
0  notgiven  original  train  
1  notgiven  original  train  
2  notgiven  original  train  
3  notgiven  original  train  
4  notgiven  original   test  


In [5]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [6]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re

# Ensure stopwords are downloaded
nltk.download('stopwords')

# Set of English stopwords
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Remove special characters and numbers, keeping only letters
    text = re.sub(r'[^A-Za-z\s]', '', text)
    # Lowercase the text
    text = text.lower()
    # Tokenize and remove stopwords
    words = word_tokenize(text)
    words = [word for word in words if word not in stop_words]
    return ' '.join(words)  # Join tokens back as a single string


# Apply the preprocessing function to the 'text' column
merged_df['cleaned_text'] = merged_df['text'].apply(preprocess_text)

# Display the first few rows to see the result
print(merged_df[['text', 'cleaned_text']].head())



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


                                                text  \
0                                dalits are lowlives   
1             gay people are a burden to our society   
2                              Arabs are not welcome   
3  I'm not saying we should actually eliminate he...   
4                       bananas are for black people   

                                        cleaned_text  
0                                    dalits lowlives  
1                          gay people burden society  
2                                      arabs welcome  
3  im saying actually eliminate heebs wish natura...  
4                               bananas black people  


In [7]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [8]:
# Load GloVe embeddings
def load_glove_embeddings(file_path):
    embeddings_index = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.array(values[1:], dtype='float32')
            embeddings_index[word] = vector
    return embeddings_index

# Load GloVe 100d embeddings
glove_embeddings = load_glove_embeddings("glove.6B.100d.txt")
embedding_dim = 100

In [9]:
# Function to get the average GloVe embedding for a sentence
def get_sentence_embedding(sentence, embeddings_index, embedding_dim):
    words = sentence.split()  # Assumes sentence is pre-tokenized (lowercase)
    valid_embeddings = [embeddings_index.get(word, np.zeros(embedding_dim)) for word in words]
    if valid_embeddings:
        return np.mean(valid_embeddings, axis=0)
    else:
        return np.zeros(embedding_dim)

merged_df['embedding'] = merged_df['cleaned_text'].apply(lambda x: get_sentence_embedding(x, glove_embeddings, embedding_dim))


In [10]:
from sklearn.model_selection import train_test_split

# Separate features and labels
X = np.vstack(merged_df['embedding'].values)  # Stack embeddings as numpy array
y = merged_df['label'].apply(lambda x: 1 if x == 'hate' else 0).values  # Convert labels to binary

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Display shapes of training and testing sets
print("Training set shape:", X_train.shape, y_train.shape)
print("Testing set shape:", X_test.shape, y_test.shape)


Training set shape: (52741, 100) (52741,)
Testing set shape: (13186, 100) (13186,)


In [20]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Create DataLoader for batching
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Define the neural network architecture
class HateSpeechClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(HateSpeechClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# Hyperparameter configurations to try
hyperparameter_history = []
configs = [
    {'hidden_dim': 50, 'batch_size': 32, 'learning_rate': 0.001, 'epochs': 50},
    {'hidden_dim': 100, 'batch_size': 64, 'learning_rate': 0.005, 'epochs': 50},
    {'hidden_dim': 150, 'batch_size': 128, 'learning_rate': 0.0001, 'epochs': 50}
]

# Iterate through hyperparameter configurations
for config in configs:
    # Set hyperparameters
    input_dim = X_train.shape[1]
    hidden_dim = config['hidden_dim']
    output_dim = 2
    batch_size = config['batch_size']
    learning_rate = config['learning_rate']
    num_epochs = config['epochs']

    # Update DataLoader with the new batch size
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # Initialize model, loss function, and optimizer
    model = HateSpeechClassifier(input_dim, hidden_dim, output_dim)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # Training loop
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct_predictions = 0
        total_predictions = 0

        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct_predictions += (predicted == labels).sum().item()
            total_predictions += labels.size(0)

    # Evaluate the model on the test set
    model.eval()
    with torch.no_grad():
        outputs = model(X_test_tensor)
        _, predicted = torch.max(outputs, 1)
        test_accuracy = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)

    # Log the results
    hyperparameter_history.append({
        'hidden_dim': hidden_dim,
        'batch_size': batch_size,
        'learning_rate': learning_rate,
        'epochs': num_epochs,
        'test_accuracy': test_accuracy
    })

    print(f"Config: {config}, Test Accuracy: {test_accuracy:.4f}")

# Display the history
print("\nHyperparameter History:")
for entry in hyperparameter_history:
    print(entry)


Config: {'hidden_dim': 50, 'batch_size': 32, 'learning_rate': 0.001, 'epochs': 50}, Test Accuracy: 0.7070
Config: {'hidden_dim': 100, 'batch_size': 64, 'learning_rate': 0.005, 'epochs': 50}, Test Accuracy: 0.6980
Config: {'hidden_dim': 150, 'batch_size': 128, 'learning_rate': 0.0001, 'epochs': 50}, Test Accuracy: 0.7229

Hyperparameter History:
{'hidden_dim': 50, 'batch_size': 32, 'learning_rate': 0.001, 'epochs': 50, 'test_accuracy': 0.7069619293189747}
{'hidden_dim': 100, 'batch_size': 64, 'learning_rate': 0.005, 'epochs': 50, 'test_accuracy': 0.6980130441377218}
{'hidden_dim': 150, 'batch_size': 128, 'learning_rate': 0.0001, 'epochs': 50, 'test_accuracy': 0.7228879114212043}


In [21]:
# Test the model on a few samples
model.eval()  # Set model to evaluation mode

# Choose some test samples to test
num_samples = 5
sample_inputs = X_test_tensor[:num_samples]
sample_labels = y_test_tensor[:num_samples]

# Run predictions
with torch.no_grad():
    outputs = model(sample_inputs)
    
    # Get the predicted class and probabilities
    _, predicted_labels = torch.max(outputs, 1)
    predicted_probabilities = torch.nn.functional.softmax(outputs, dim=1)
    
    for i in range(num_samples):
        true_label = sample_labels[i].item()
        predicted_label = predicted_labels[i].item()
        probabilities = predicted_probabilities[i].numpy()
        
        print(f"Sample {i+1}:")
        print(f"True Label: {'hate' if true_label == 1 else 'nothate'}")
        print(f"Predicted Label: {'hate' if predicted_label == 1 else 'nothate'}")
        print(f"Predicted Probabilities: Hate: {probabilities[1]:.4f}, Not Hate: {probabilities[0]:.4f}")
        print("-" * 40)




Sample 1:
True Label: nothate
Predicted Label: nothate
Predicted Probabilities: Hate: 0.0798, Not Hate: 0.9202
----------------------------------------
Sample 2:
True Label: hate
Predicted Label: hate
Predicted Probabilities: Hate: 0.7084, Not Hate: 0.2916
----------------------------------------
Sample 3:
True Label: nothate
Predicted Label: nothate
Predicted Probabilities: Hate: 0.1326, Not Hate: 0.8674
----------------------------------------
Sample 4:
True Label: nothate
Predicted Label: nothate
Predicted Probabilities: Hate: 0.3764, Not Hate: 0.6236
----------------------------------------
Sample 5:
True Label: hate
Predicted Label: hate
Predicted Probabilities: Hate: 0.6743, Not Hate: 0.3257
----------------------------------------


In [22]:
# Save the trained PyTorch model
torch.save(model.state_dict(), "hate_speech_model.pth")
